# Iris ML Demo: MLflow, AutoML, and Hyperopt

This notebook demonstrates a complete machine learning workflow using:
- **MLflow** for experiment tracking and model management
- **Databricks AutoML** for automated model selection
- **Hyperopt** for hyperparameter tuning

**Dataset**: Unity Catalog table `main.tomes_gen.iris`  
**Target**: `Species` (multi-class classification)  
**Date**: November 2025


## Phase 1: Setup and Data Preparation


### Step 1.1: Import Required Libraries


In [ ]:
# System imports
import sys
import os
sys.path.append(os.path.abspath(".."))

# Spark imports
from spark_env import spark
from pyspark.sql.functions import col, lit
import pyspark.sql.functions as F

# MLflow
import mlflow

# Databricks AutoML
# import databricks.automl

# Scikit-learn (for Hyperopt tuning)
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# LightGBM (for Hyperopt tuning)
# import lightgbm as lgb
from lightgbm import LGBMClassifier

# Hyperopt
from hyperopt import fmin, tpe, hp, SparkTrials, STATUS_OK, Trials
from hyperopt.pyll import scope

# NumPy and Pandas
import numpy as np
import pandas as pd

print("All libraries imported successfully!")


Driver is running on local environment


ModuleNotFoundError: No module named 'databricks.automl'

### Step 1.2: Load Data from Unity Catalog


In [ ]:
# Load the dataset from Unity Catalog
df = spark.table("main.tomes_gen.iris")

# Display basic information
print("Dataset shape (rows, columns):")
print(f"Rows: {df.count()}, Columns: {len(df.columns)}")

print("\nFirst few rows:")
df.show(10)

print("\nSchema:")
df.printSchema()

print("\nSummary statistics:")
df.describe().show()

# Verify target column and class distribution
print("\nClass distribution (Species):")
df.groupBy("Species").count().orderBy("Species").show()


### Step 1.3: Data Preprocessing (Optional for AutoML)

Databricks AutoML handles most preprocessing automatically, including:
- Missing value imputation
- Feature engineering
- Train/validation/test splitting

We just need to verify the target column is correctly identified and exclude any ID columns if present.


In [ ]:
# Check for ID column to exclude
id_cols = ["Id", "id", "ID"] if any(col in df.columns for col in ["Id", "id", "ID"]) else None
exclude_cols = [col for col in id_cols if col in df.columns] if id_cols else None

print(f"Columns to exclude from features: {exclude_cols}")
print(f"Target column: Species")
print(f"Feature columns: {[c for c in df.columns if c != 'Species' and (exclude_cols is None or c not in exclude_cols)]}")

# Check for missing values (AutoML will handle imputation automatically)
print("\nMissing values per column:")
for col_name in df.columns:
    missing_count = df.filter(F.col(col_name).isNull()).count()
    if missing_count > 0:
        print(f"  {col_name}: {missing_count}")
    else:
        print(f"  {col_name}: 0 (no missing values)")


## Phase 2: MLflow Configuration


### Step 2.1: Initialize MLflow

MLflow autologging will automatically track:
- Parameters
- Metrics
- Model artifacts
- Model signatures


In [ ]:
# Set MLflow tracking URI to Databricks
mlflow.set_tracking_uri("databricks")

# Set experiment name
experiment_name = "iris_demo"
mlflow.set_experiment(experiment_name)

# Enable autologging for scikit-learn models
mlflow.sklearn.autolog()

# Verify connection
print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment: {mlflow.get_experiment_by_name(experiment_name)}")
print("MLflow autologging enabled for scikit-learn")


## Phase 3: Databricks AutoML Model Selection

Databricks AutoML will automatically:
- Test multiple algorithms (sklearn, LightGBM, potentially XGBoost)
- Perform hyperparameter tuning
- Split data into train/validation/test sets
- Log all trials to MLflow
- Generate trial notebooks for the best model


### Step 3.2: Run Databricks AutoML Classification


In [ ]:
# Run Databricks AutoML classification
# AutoML will test multiple algorithms and log all runs to MLflow
summary = databricks.automl.classify(
    dataset=df,  # Spark DataFrame
    target_col="Species",
    primary_metric="f1",  # Options: "f1", "log_loss", "precision", "accuracy", "roc_auc"
    experiment_name="iris_demo",  # Align with Phase 2 experiment
    exclude_cols=exclude_cols,  # Exclude ID column if present
    timeout_minutes=30,  # Adjust based on time constraints (default is 120)
)

print("AutoML run completed!")
print(f"Number of trials: {len(summary.trials)}")


### Step 3.3: Analyze AutoML Results


In [ ]:
# Access the best trial
best_trial = summary.best_trial

print("=" * 60)
print("AutoML Summary")
print("=" * 60)
print(f"Total trials: {len(summary.trials)}")
print(f"\nBest Trial Information:")
print(f"  Evaluation metric score (F1): {best_trial.evaluation_metric_score:.4f}")
print(f"  Model description: {best_trial.model_description}")
print(f"\nBest Trial Metrics:")
for key, value in best_trial.metrics.items():
    print(f"  {key}: {value:.4f}")
print(f"\nBest Trial Parameters:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")

# Load the best model
best_model = best_trial.load_model()
print(f"\nBest model type: {type(best_model).__name__}")

# Identify model type for Hyperopt tuning
# Extract model type from model description or class name
model_type = type(best_model).__name__
if "RandomForest" in model_type or "random_forest" in str(best_trial.model_description).lower():
    best_model_type = "RandomForest"
elif "LightGBM" in model_type or "lightgbm" in str(best_trial.model_description).lower() or "LGBM" in model_type:
    best_model_type = "LightGBM"
elif "SVM" in model_type or "SVC" in model_type or "svm" in str(best_trial.model_description).lower():
    best_model_type = "SVM"
elif "LogisticRegression" in model_type or "logistic" in str(best_trial.model_description).lower():
    best_model_type = "LogisticRegression"
elif "DecisionTree" in model_type or "decision" in str(best_trial.model_description).lower():
    best_model_type = "DecisionTree"
else:
    # Default fallback - try to infer from model class
    best_model_type = model_type

print(f"\nIdentified model type for Hyperopt tuning: {best_model_type}")


**AutoML Results Summary:**

The best model identified by AutoML will be displayed above. This model will be further tuned using Hyperopt in the next phase to potentially improve performance.


## Phase 4: Hyperparameter Tuning with Hyperopt

**Important**: We will only tune the best model identified by AutoML (not all models). This allows us to focus computational resources on the most promising model architecture.


### Step 4.1: Prepare Data for Hyperopt


In [ ]:
# Load the original dataset and convert to Pandas for Hyperopt
iris_df = df.toPandas()

# Separate features and target
# Exclude ID column if present and the target column
feature_cols = [c for c in iris_df.columns if c != 'Species' and (exclude_cols is None or c not in exclude_cols)]
X = iris_df[feature_cols]
y = iris_df['Species']

print(f"Feature columns: {feature_cols}")
print(f"Target column: Species")
print(f"Shape: X={X.shape}, y={y.shape}")

# Encode target labels if needed (convert string labels to numeric)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
label_mapping = dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))
print(f"\nLabel encoding mapping: {label_mapping}")

# Create train/test split
# Use stratify to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_encoded
)

print(f"\nTrain set: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Test set: X_test={X_test.shape}, y_test={y_test.shape}")
print(f"Train class distribution: {np.bincount(y_train)}")
print(f"Test class distribution: {np.bincount(y_test)}")


### Step 4.2: Define Objective Function


In [ ]:
# Define objective function for Hyperopt
# This function will be called for each hyperparameter trial
def objective(params):
    """
    Objective function for Hyperopt optimization.
    Creates a model with given hyperparameters, trains it, and evaluates it.
    """
    # Create a new MLflow run for this trial
    with mlflow.start_run(nested=True, run_name="hyperopt_trial"):
        try:
            # Instantiate model based on best_model_type
            if best_model_type == "RandomForest":
                model = RandomForestClassifier(
                    n_estimators=int(params['n_estimators']),
                    max_depth=int(params['max_depth']),
                    min_samples_split=int(params['min_samples_split']),
                    random_state=42,
                    n_jobs=-1
                )
            elif best_model_type == "LightGBM":
                model = LGBMClassifier(
                    n_estimators=int(params['n_estimators']),
                    max_depth=int(params['max_depth']),
                    learning_rate=params['learning_rate'],
                    num_leaves=int(params['num_leaves']),
                    min_child_samples=int(params['min_child_samples']),
                    subsample=params['subsample'],
                    colsample_bytree=params['colsample_bytree'],
                    random_state=42,
                    verbose=-1
                )
            elif best_model_type == "SVM":
                kernel = params['kernel']
                model_params = {
                    'C': params['C'],
                    'kernel': kernel,
                    'random_state': 42
                }
                if kernel != 'linear':
                    model_params['gamma'] = params['gamma']
                model = SVC(**model_params)
            elif best_model_type == "LogisticRegression":
                penalty = params['penalty']
                solver_map = {
                    'l1': 'liblinear',
                    'l2': 'lbfgs',
                    'elasticnet': 'saga'
                }
                model = LogisticRegression(
                    C=params['C'],
                    penalty=penalty,
                    solver=solver_map[penalty],
                    random_state=42,
                    max_iter=1000
                )
            elif best_model_type == "DecisionTree":
                model = DecisionTreeClassifier(
                    max_depth=int(params['max_depth']),
                    min_samples_split=int(params['min_samples_split']),
                    min_samples_leaf=int(params['min_samples_leaf']),
                    random_state=42
                )
            else:
                # Fallback: try to use the model type directly
                raise ValueError(f"Unsupported model type: {best_model_type}")
            
            # Train the model
            model.fit(X_train, y_train)
            
            # Make predictions
            y_pred = model.predict(X_test)
            
            # Evaluate on test set
            accuracy = accuracy_score(y_test, y_pred)
            
            # Log metrics to MLflow
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_params(params)
            
            # Return dictionary for Hyperopt
            return {
                'loss': -accuracy,  # Negative because Hyperopt minimizes
                'status': STATUS_OK,
                'accuracy': accuracy
            }
            
        except Exception as e:
            # Handle invalid hyperparameter combinations
            print(f"Error in trial: {e}")
            return {
                'loss': 1.0,  # High loss for failed trials
                'status': STATUS_OK,
                'error': str(e)
            }

print("Objective function defined successfully!")


### Step 4.3: Define Hyperparameter Search Space


In [ ]:
# Define search space based on the best model type identified by AutoML
if best_model_type == "SVM":
    search_space = {
        'C': hp.loguniform('C', -5, 5),
        'kernel': hp.choice('kernel', ['linear', 'rbf', 'poly']),
        'gamma': hp.loguniform('gamma', -5, 5)
    }
elif best_model_type == "LogisticRegression":
    search_space = {
        'C': hp.loguniform('C', -5, 5),
        'penalty': hp.choice('penalty', ['l1', 'l2', 'elasticnet'])
    }
elif best_model_type == "DecisionTree":
    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 3, 20, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 20, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 10, 1))
    }
elif best_model_type == "RandomForest":
    search_space = {
        'n_estimators': scope.int(hp.quniform('n_estimators', 50, 200, 10)),
        'max_depth': scope.int(hp.quniform('max_depth', 3, 20, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 20, 1))
    }
elif best_model_type == "LightGBM":
    search_space = {
        'n_estimators': scope.int(hp.quniform('n_estimators', 50, 300, 10)),
        'max_depth': scope.int(hp.quniform('max_depth', 3, 15, 1)),
        'learning_rate': hp.loguniform('learning_rate', -3, 0),  # 0.001 to 1
        'num_leaves': scope.int(hp.quniform('num_leaves', 20, 300, 10)),
        'min_child_samples': scope.int(hp.quniform('min_child_samples', 5, 100, 5)),
        'subsample': hp.uniform('subsample', 0.6, 1.0),
        'colsample_bytree': hp.uniform('colsample_bytree', 0.6, 1.0)
    }
else:
    raise ValueError(f"Unsupported model type for Hyperopt: {best_model_type}")

print(f"Hyperparameter search space defined for {best_model_type}:")
print(f"  Parameters: {list(search_space.keys())}")


### Step 4.4: Configure SparkTrials for Parallel Execution


In [ ]:
# Create SparkTrials object for parallel hyperparameter search
# This enables parallel execution across Spark executors
spark_trials = SparkTrials(parallelism=4)  # Adjust based on cluster capacity

print(f"SparkTrials configured with parallelism={spark_trials.parallelism}")
print("This will enable parallel hyperparameter search across Spark executors")


### Step 4.5: Run Hyperopt Optimization


In [ ]:
# Run Hyperopt optimization
# Each trial will create its own MLflow run, so all tuning attempts are tracked
print("Starting Hyperopt optimization...")
print(f"Model type: {best_model_type}")
print(f"Max evaluations: 50")
print(f"Parallelism: {spark_trials.parallelism}")

with mlflow.start_run(run_name="hyperopt_optimization"):
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,  # Tree-structured Parzen Estimator algorithm
        max_evals=50,  # Adjust based on time constraints
        trials=spark_trials
    )

print("\n" + "=" * 60)
print("Hyperopt Optimization Complete")
print("=" * 60)
print(f"Best parameters found:")
for key, value in best_params.items():
    print(f"  {key}: {value}")

# Get the best loss (which is negative accuracy)
best_loss = spark_trials.best_trial['result']['loss']
best_accuracy = -best_loss  # Convert back to accuracy
print(f"\nBest accuracy from Hyperopt: {best_accuracy:.4f}")


### Step 4.6: Train Final Model with Best Parameters


In [ ]:
# Train final model with best hyperparameters from Hyperopt
with mlflow.start_run(run_name="best_tuned_model"):
    # Handle parameter conversion for choice parameters (Hyperopt returns indices)
    # Also prepare parameters for logging
    if best_model_type == "SVM":
        kernel_choices = ['linear', 'rbf', 'poly']
        best_kernel = kernel_choices[best_params['kernel']]
        final_params = {
            'C': best_params['C'],
            'kernel': best_kernel
        }
        if best_kernel != 'linear':
            final_params['gamma'] = best_params['gamma']
        final_model = SVC(**final_params, random_state=42)
        log_params = final_params
        
    elif best_model_type == "LogisticRegression":
        penalty_choices = ['l1', 'l2', 'elasticnet']
        best_penalty = penalty_choices[best_params['penalty']]
        solver_map = {
            'l1': 'liblinear',
            'l2': 'lbfgs',
            'elasticnet': 'saga'
        }
        final_model = LogisticRegression(
            C=best_params['C'],
            penalty=best_penalty,
            solver=solver_map[best_penalty],
            random_state=42,
            max_iter=1000
        )
        log_params = {
            'C': best_params['C'],
            'penalty': best_penalty
        }
        
    elif best_model_type == "DecisionTree":
        final_model = DecisionTreeClassifier(
            max_depth=int(best_params['max_depth']),
            min_samples_split=int(best_params['min_samples_split']),
            min_samples_leaf=int(best_params['min_samples_leaf']),
            random_state=42
        )
        log_params = {
            'max_depth': int(best_params['max_depth']),
            'min_samples_split': int(best_params['min_samples_split']),
            'min_samples_leaf': int(best_params['min_samples_leaf'])
        }
        
    elif best_model_type == "RandomForest":
        final_model = RandomForestClassifier(
            n_estimators=int(best_params['n_estimators']),
            max_depth=int(best_params['max_depth']),
            min_samples_split=int(best_params['min_samples_split']),
            random_state=42,
            n_jobs=-1
        )
        log_params = {
            'n_estimators': int(best_params['n_estimators']),
            'max_depth': int(best_params['max_depth']),
            'min_samples_split': int(best_params['min_samples_split'])
        }
        
    elif best_model_type == "LightGBM":
        final_model = LGBMClassifier(
            n_estimators=int(best_params['n_estimators']),
            max_depth=int(best_params['max_depth']),
            learning_rate=best_params['learning_rate'],
            num_leaves=int(best_params['num_leaves']),
            min_child_samples=int(best_params['min_child_samples']),
            subsample=best_params['subsample'],
            colsample_bytree=best_params['colsample_bytree'],
            random_state=42,
            verbose=-1
        )
        log_params = {
            'n_estimators': int(best_params['n_estimators']),
            'max_depth': int(best_params['max_depth']),
            'learning_rate': best_params['learning_rate'],
            'num_leaves': int(best_params['num_leaves']),
            'min_child_samples': int(best_params['min_child_samples']),
            'subsample': best_params['subsample'],
            'colsample_bytree': best_params['colsample_bytree']
        }
    
    # Train on full training set
    final_model.fit(X_train, y_train)
    
    # Evaluate on test set
    y_pred_final = final_model.predict(X_test)
    final_accuracy = accuracy_score(y_test, y_pred_final)
    
    # Log final model and metrics
    mlflow.log_params(log_params)
    mlflow.log_metric("accuracy", final_accuracy)
    
    # Log the model
    mlflow.sklearn.log_model(final_model, "model")
    
    print(f"Final model trained with accuracy: {final_accuracy:.4f}")
    print(f"Model logged to MLflow run: {mlflow.active_run().info.run_id}")
    
    # Compare with AutoML best model
    automl_accuracy = best_trial.metrics.get('val_f1_score', best_trial.evaluation_metric_score)
    print(f"\nComparison:")
    print(f"  AutoML best model F1 score: {automl_accuracy:.4f}")
    print(f"  Hyperopt-tuned model accuracy: {final_accuracy:.4f}")
    print(f"  Improvement: {final_accuracy - automl_accuracy:.4f}")


## Phase 5: Model Evaluation and Summary


### Step 5.1: Final Model Evaluation


In [ ]:
# Generate detailed evaluation metrics
print("=" * 60)
print("Final Model Evaluation")
print("=" * 60)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred_final)
print("\nConfusion Matrix:")
print(cm)

# Classification report
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=label_encoder.classes_))

# Per-class metrics
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(y_test, y_pred_final, average=None)
recall = recall_score(y_test, y_pred_final, average=None)
f1 = f1_score(y_test, y_pred_final, average=None)

print("\nPer-class Metrics:")
for i, class_name in enumerate(label_encoder.classes_):
    print(f"  {class_name}:")
    print(f"    Precision: {precision[i]:.4f}")
    print(f"    Recall: {recall[i]:.4f}")
    print(f"    F1-Score: {f1[i]:.4f}")

# Overall metrics
print(f"\nOverall Metrics:")
print(f"  Accuracy: {final_accuracy:.4f}")
print(f"  Macro Precision: {precision_score(y_test, y_pred_final, average='macro'):.4f}")
print(f"  Macro Recall: {recall_score(y_test, y_pred_final, average='macro'):.4f}")
print(f"  Macro F1-Score: {f1_score(y_test, y_pred_final, average='macro'):.4f}")


In [ ]:
# Visualize confusion matrix (optional but recommended for demo)
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=label_encoder.classes_, 
                yticklabels=label_encoder.classes_)
    plt.title('Confusion Matrix - Final Tuned Model')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()
    
    # Feature importance (if applicable to model type)
    if hasattr(final_model, 'feature_importances_'):
        importances = final_model.feature_importances_
        feature_importance_df = pd.DataFrame({
            'feature': feature_cols,
            'importance': importances
        }).sort_values('importance', ascending=False)
        
        plt.figure(figsize=(8, 6))
        sns.barplot(data=feature_importance_df, x='importance', y='feature')
        plt.title('Feature Importance - Final Tuned Model')
        plt.xlabel('Importance')
        plt.tight_layout()
        plt.show()
        
        print("\nFeature Importance:")
        print(feature_importance_df.to_string(index=False))
    else:
        print("\nFeature importance not available for this model type.")
        
except ImportError:
    print("Matplotlib/Seaborn not available for visualization.")
except Exception as e:
    print(f"Error creating visualization: {e}")


### Step 5.2: MLflow Model Registration (Optional but Recommended)


In [ ]:
# Register the best model in MLflow Model Registry
# Note: This step is optional but recommended for production workflows

try:
    # The model was already logged in the previous cell
    # Now we can register it
    model_uri = f"runs:/{mlflow.active_run().info.run_id}/model"
    
    # Register the model
    registered_model = mlflow.register_model(
        model_uri=model_uri,
        name="iris_classifier"
    )
    
    print(f"Model registered successfully!")
    print(f"  Model name: {registered_model.name}")
    print(f"  Model version: {registered_model.version}")
    print(f"  Model stage: {registered_model.current_stage}")
    
except Exception as e:
    print(f"Model registration skipped or failed: {e}")
    print("You can register the model manually from the MLflow UI if needed.")


### Step 5.3: Summary and Documentation

## Summary

This notebook demonstrated a complete machine learning workflow using MLflow, Databricks AutoML, and Hyperopt:

### Workflow Overview:
1. **Data Preparation**: Loaded Iris dataset from Unity Catalog (`main.tomes_gen.iris`)
2. **MLflow Setup**: Configured experiment tracking with autologging enabled
3. **AutoML Model Selection**: Used Databricks AutoML to automatically test multiple algorithms and select the best model
4. **Hyperparameter Tuning**: Applied Hyperopt to further tune the best AutoML model
5. **Model Evaluation**: Evaluated the final tuned model with comprehensive metrics

### Key Learnings:

1. **Databricks AutoML** provides an excellent starting point by automatically testing multiple algorithms and hyperparameter combinations
2. **Hyperopt** with SparkTrials enables efficient parallel hyperparameter tuning
3. **MLflow** tracks all experiments, making it easy to compare models and reproduce results
4. The combination of AutoML + Hyperopt provides both breadth (algorithm selection) and depth (fine-tuning)

### Next Steps:

- Review all runs in the MLflow UI to compare different models
- Consider deploying the registered model for production use
- Experiment with different primary metrics or search spaces for further optimization

**Note**: The actual model type, accuracy, and metrics are displayed in the cells above. Review those results for the specific values from your run.
